# PS5 Textile Defect Detection – Training on Google Colab

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Your Google Drive must look like this (create the folders yourself):

```
MyDrive/
  PS5/
    dataset_zips/
      det_part1_of_4.zip ... det_part4_of_4.zip      <- detection (60% defect-free / 40% defect)
      cls_part1a_train.zip, cls_part1b_train.zip, cls_part2_val_test.zip   <- classification
      own_photos.zip      <- OPTIONAL: your own photos exported from Roboflow (format "YOLOv8")
```

Run the cells top to bottom. Trained models and results are saved to `MyDrive/PS5/runs`
and `MyDrive/PS5/models`, so nothing is lost if Colab disconnects.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Settings (change only if your folder names are different)

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/PS5'          # your project folder in Drive
ZIPS  = f'{DRIVE}/dataset_zips'               # where the zip files are
DATA  = '/content/textile_dataset'            # fast local copy used for training
OWN_ZIP = f'{ZIPS}/own_photos.zip'            # Roboflow export of your own photos (optional)

# Roboflow class name  ->  classification folder name
CLASS_MAP = {
    'hole': 'hole_burst', 'burst': 'hole_burst',
    'stain': 'stain_oil', 'oil_stain': 'stain_oil', 'oil stain': 'stain_oil',
    'broken_yarn': 'broken_yarn_needle', 'broken yarn': 'broken_yarn_needle', 'needle_break': 'broken_yarn_needle',
    'lycra_run': 'lycra_run', 'jute': 'jute',
}
os.makedirs(f'{DRIVE}/runs', exist_ok=True); os.makedirs(f'{DRIVE}/models', exist_ok=True)
print(sorted(os.listdir(ZIPS)))

## 4. Unzip the dataset from Drive into Colab
All parts unzip into one folder and join up automatically.

In [ ]:
import glob, zipfile, os
os.makedirs(DATA, exist_ok=True)
zips = sorted(glob.glob(f'{ZIPS}/det_part*.zip') + glob.glob(f'{ZIPS}/cls_part*.zip'))
assert zips, 'No dataset zips found - check the ZIPS folder path'
for z in zips:
    print('unzipping', os.path.basename(z))
    zipfile.ZipFile(z).extractall(DATA)

for s in ['train', 'val', 'test']:
    n_det = len(os.listdir(f'{DATA}/detection/images/{s}'))
    n_cls = sum(len(os.listdir(f'{DATA}/classification/{s}/{c}')) for c in os.listdir(f'{DATA}/classification/{s}'))
    print(f'{s:5s}  detection: {n_det:6d}   classification: {n_cls:6d}')

## 5. Combine your own photos (optional)
Label your photos in Roboflow with boxes (classes like `hole`, `stain`, `broken_yarn`), export as **YOLOv8**,
rename the zip to `own_photos.zip` and put it in `dataset_zips`. Photos of good fabric with no boxes are fine too.

This cell:
- adds every own photo to the **detection** set (all classes become `defect`), keeping Roboflow's train/valid/test split
- cuts out each labelled box and adds it to the matching **classification** folder (using `CLASS_MAP`)

In [ ]:
import zipfile, os, shutil, yaml, glob
from PIL import Image

def add_own_photos(own_zip, data, class_map):
    if not os.path.exists(own_zip):
        print('No own_photos.zip found - skipping (that is OK)'); return
    tmp = '/content/own_tmp'; shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(own_zip).extractall(tmp)
    yml = glob.glob(f'{tmp}/**/data.yaml', recursive=True)[0]; base = os.path.dirname(yml)
    names = yaml.safe_load(open(yml))['names']
    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
    added = {'detection': 0, 'crops': 0, 'skipped_classes': set()}
    for rf_split, split in [('train', 'train'), ('valid', 'val'), ('test', 'test')]:
        for img in glob.glob(f'{base}/{rf_split}/images/*'):
            stem = os.path.splitext(os.path.basename(img))[0]
            lab = f'{base}/{rf_split}/labels/{stem}.txt'
            lines = [l.split() for l in open(lab)] if os.path.exists(lab) else []
            lines = [l for l in lines if len(l) == 5]
            im = Image.open(img).convert('RGB')
            new = f'own_{stem}'
            im.save(f'{data}/detection/images/{split}/{new}.jpg', quality=95)
            with open(f'{data}/detection/labels/{split}/{new}.txt', 'w') as f:
                for l in lines: f.write('0 ' + ' '.join(l[1:]) + '\n')
            added['detection'] += 1
            W, H = im.size
            for i, l in enumerate(lines):
                name = str(names[int(l[0])]).strip().lower()
                folder = class_map.get(name)
                if folder is None: added['skipped_classes'].add(name); continue
                x, y, w, h = map(float, l[1:])
                pad = 0.15  # keep a little fabric around the defect
                x0, y0 = max(0, (x - w/2 - pad*w) * W), max(0, (y - h/2 - pad*h) * H)
                x1, y1 = min(W, (x + w/2 + pad*w) * W), min(H, (y + h/2 + pad*h) * H)
                if x1 - x0 < 16 or y1 - y0 < 16: continue
                d = f'{data}/classification/{split}/{folder}'; os.makedirs(d, exist_ok=True)
                im.crop((x0, y0, x1, y1)).save(f'{d}/{new}_{i}.jpg', quality=95)
                added['crops'] += 1
    print(f"Added {added['detection']} own photos to detection and {added['crops']} crops to classification")
    if added['skipped_classes']: print('Class names not in CLASS_MAP (not added to classification):', added['skipped_classes'])

add_own_photos(OWN_ZIP, DATA, CLASS_MAP)

# point data.yaml at the local folder
open(f'{DATA}/detection/data.yaml', 'w').write(
    f'path: {DATA}/detection\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: defect\n')

## 6. Check the 60% defect-free / 40% defect balance

In [ ]:
import os
for s in ['train', 'val', 'test']:
    d = f'{DATA}/detection/labels/{s}'
    labs = os.listdir(d)
    defect = sum(os.path.getsize(f'{d}/{l}') > 0 for l in labs); free = len(labs) - defect
    pct = 100 * free / len(labs)
    flag = 'OK' if 55 <= pct <= 65 else 'CHECK: add more ' + ('defect-free' if pct < 55 else 'defect') + ' photos'
    print(f'{s:5s}: {len(labs):6d} images | defect-free {free:6d} ({pct:4.1f}%) | defect {defect:6d} ({100-pct:4.1f}%) | {flag}')

## 7. Install Ultralytics

In [ ]:
!pip install -q ultralytics

## 8. Train the detector (finds WHERE the defect is)
About 1–2 hours on a T4. If Colab disconnects, run the **resume** cell below instead of starting again.
If `yolo26n.pt` gives an error, change it to `yolo11n.pt`.

In [ ]:
from ultralytics import YOLO
det = YOLO('yolo26n.pt')
det.train(data=f'{DATA}/detection/data.yaml', imgsz=512, epochs=60, batch=32,
          project=f'{DRIVE}/runs', name='detector', exist_ok=True, patience=15)

In [ ]:
# RESUME only if training was interrupted
# from ultralytics import YOLO
# YOLO(f'{DRIVE}/runs/detector/weights/last.pt').train(resume=True)

## 9. Test the detector on the test split (numbers for your report)

In [ ]:
from ultralytics import YOLO
det = YOLO(f'{DRIVE}/runs/detector/weights/best.pt')
m = det.val(data=f'{DATA}/detection/data.yaml', split='test', imgsz=512, project=f'{DRIVE}/runs', name='detector_test', exist_ok=True)
print('mAP50:', round(m.box.map50, 3), ' mAP50-95:', round(m.box.map, 3), ' precision:', round(m.box.mp, 3), ' recall:', round(m.box.mr, 3))

## 10. Train the defect-type classifier (says WHAT the defect is)

In [ ]:
from ultralytics import YOLO
cls = YOLO('yolo26n-cls.pt')
cls.train(data=f'{DATA}/classification', imgsz=224, epochs=40, batch=64,
          project=f'{DRIVE}/runs', name='classifier', exist_ok=True, patience=10)

In [ ]:
from ultralytics import YOLO
cls = YOLO(f'{DRIVE}/runs/classifier/weights/best.pt')
r = cls.val(data=f'{DATA}/classification', split='test', imgsz=224, project=f'{DRIVE}/runs', name='classifier_test', exist_ok=True)
print('top-1 accuracy:', round(r.top1, 3))

## 11. Export both models to ExecuTorch for the Raspberry Pi

In [ ]:
import shutil, glob, os
from ultralytics import YOLO
for name, size in [('detector', 512), ('classifier', 224)]:
    out = YOLO(f'{DRIVE}/runs/{name}/weights/best.pt').export(format='executorch', imgsz=size)
    print(name, '->', out)
    dst = f'{DRIVE}/models/{name}_executorch'
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(out, dst) if os.path.isdir(out) else shutil.copy(out, f'{DRIVE}/models/')
print(os.listdir(f'{DRIVE}/models'))

## 12. Quick visual check on a few test images

In [ ]:
import glob, random
from ultralytics import YOLO
from IPython.display import Image, display
det = YOLO(f'{DRIVE}/runs/detector/weights/best.pt')
imgs = random.sample(glob.glob(f'{DATA}/detection/images/test/*.jpg'), 6)
res = det.predict(imgs, imgsz=512, conf=0.25, save=True, project='/content/preds', name='check', exist_ok=True)
for p in glob.glob('/content/preds/check/*.jpg'): display(Image(filename=p, width=320))